# Question 5: Opinion Dynamics in Facebook100

This notebook analyzes opinion dynamics across 100 Facebook university networks, following the specifications of Question 5 from the problem set.

### Question Text:
> Download the Facebook100 data, containing facebook networks for 100 universities in the US. We want to analyze how the different networks generate different opinion dynamics. Keep the largest component of each network only.
>
> (a) To simplify things we will assume that each node "listens" to all her neighbors (and herself) with the same probability. In other words, we assume that $T_{ij} = 1/(1 + d_i)$ for all $j \in N_i$ and also $T_{ii} = 1/(1 + d_i)$.
>
> (b) Since we don’t have any way of performing sentiment analysis on this data, we will instead simulate innate/original opinions from a $Uniform(0,1)$. In other words, assign to each node a $p_i(0)$ that is drawn from a uniform random variable.
>
> (c) Since we have only one component and all edges are bi-directional (and we assume everyone pays some attention to themselves) then we should obtain convergence to consensus. Compute the influence vector for each network and report a histogram of the influences obtained across all vectors. What is the maximum influence? and the median influence?
>
> (d) Plot a histogram of the consensus opinions across all networks. Which is the most popular (i.e. modal) opinion?
>
> (e) Now we want to test the speed of learning. For each network, calculate the first period, $t^\star$, where $\sum_i \tilde{d}_i(A) (p_i(t) - p_i(\infty))^2 < \epsilon$ for $\epsilon = 0.1$ and $\tilde{d}_i(A) = \frac{d_i(A)}{\sum_i d_i(A)}$.
>
> (f) Construct a plot with the second eigenvalue of the network on the x-axis and $t^\star$ on the y-axis. Do you get the relationship you expected? What happens as you lower the value of $\epsilon$?
>
> (g) (Extra points) Now assume that the $p_i(0)$ that you obtained from the uniform distribution above is in fact the innate opinion $s_i$ of agent $i$. Simulate a susceptibility parameter $\alpha_i$ also from the $Uniform(0,1)$ for each agent $i$ and solve, for each network, the steady-state value $p^\star$ under the Friedkin-Johnsen model. Construct a histogram of $p^\star$ across all networks. What is the most popular opinion? How does your answer change of $\alpha_i$ is distributed $Uniform(0.5, 1)$?
>
> (h) (Extra points) Pick your favorite network from Facebook100 and find the optimal vector $\alpha^\star$ that maximizes the sum of opinions $\mathbf{1}'\mathbf{p}^\star$ as in Abebe et al. (2018).


In [ ]:
import os
import scipy.io
import networkx as nx
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigs, spsolve
import matplotlib.pyplot as plt

folder = 'data/facebook_small'
files = [f for f in os.listdir(folder) if f.endswith('.mat')]
print(f"Found {len(files)} network files.")


## Processing Networks & (a) Transition Matrix
For each network, we:
1.  Load the adjacency matrix $A$.
2.  Extract the Largest Connected Component (LCC) to ensure the graph is connected, which is a prerequisite for consensus in this model.
3.  Construct the transition matrix $T$ as specified: $T_{ij} = 1/(1 + d_i)$ for neighbors and self. This is equivalent to $T = \tilde{D}^{-1}\tilde{A}$ where $\tilde{A} = A + I$ and $\tilde{D}$ is the degree matrix of $\tilde{A}$.
4.  

(b) in code at the end

In [ ]:
# Results storage
all_influences = []
consensus_opinions = []
t_stars = []
lambda_2s = []
fj_p_stars_u1 = []
fj_p_stars_u2 = []

chosen_network_data = None
chosen_network_name = "Reed98.mat"

np.random.seed(42)

for i, file in enumerate(files):
    if i % 10 == 0:
        print(f"Processing {i}/{len(files)}...")
        
    data = scipy.io.loadmat(os.path.join(folder, file))
    A = data['A']
    G = nx.from_scipy_sparse_array(A)

    # (a) LCC and Matrix Construction
    components = sorted(nx.connected_components(G), key=len, reverse=True)
    lcc_nodes = list(components[0])
    G = G.subgraph(lcc_nodes).copy()
    n = G.number_of_nodes()

    A_lcc = nx.adjacency_matrix(G)
    A_tilde = A_lcc + sp.eye(n)
    degrees_tilde = np.array(A_tilde.sum(axis=1)).flatten()
    
    T = sp.diags(1.0 / degrees_tilde).dot(A_tilde)

    # (c) Influence vector w
    # w is the stationary distribution. Since our transition is w_i = d_tilde_i / sum(d_tilde)
    w = degrees_tilde / degrees_tilde.sum()
    all_influences.extend(w.tolist())

    # (b) Simulate innate opinions U(0,1)
    p_0 = np.random.uniform(0, 1, n)
    p_inf = np.dot(w, p_0)
    consensus_opinions.append(p_inf)

    # (e) Learning Speed t*
    eps = 0.1
    pt = p_0.copy()
    t_star = 0
    for t in range(500):
        # Weighted variance check
        var = np.sum(w * (pt - p_inf)**2) #squared difference of opinions from consensus, weighted by influence
        if var < eps:
            t_star = t #convergence to same opinion within threshold
            break
        pt = T.dot(pt) # sum product of opinions with transition matrix to get next opinions
    else:
        t_star = 500
    t_stars.append(t_star)

    # (f) Second eigenvalue
    try:
        if n > 2:
            vals, _ = eigs(T, k=2, which='LM', tol=1e-2)
            eigenvalues = np.sort(np.abs(vals))
            lambda_2s.append(eigenvalues[-2])
        else:
            lambda_2s.append(0)
    except:
        lambda_2s.append(0)

    # (g) Friedkin-Johnsen steady state
    I = sp.eye(n)
    # alpha ~ U(0,1)
    alpha1 = np.random.uniform(0, 1, n)
    M1 = I - (I - sp.diags(alpha1)).dot(T)
    p_star_1 = spsolve(M1, alpha1 * p_0)
    fj_p_stars_u1.extend(p_star_1.tolist())
        
    # alpha ~ U(0.5,1)
    alpha2 = np.random.uniform(0.5, 1, n)
    M2 = I - (I - sp.diags(alpha2)).dot(T)
    p_star_2 = spsolve(M2, alpha2 * p_0)
    fj_p_stars_u2.extend(p_star_2.tolist())
        
    if file == chosen_network_name:
        chosen_network_data = {'T': T, 'p_0': p_0, 'n': n, 'name': file}

print("Finished processing all networks.")


## (c) Influence Vector

The histogram of influence values shows a highly skewed distribution. Most nodes have extremely small influence, while a small number of nodes have significantly larger influence. This reflects the heterogeneity of the network structure.

Quantitatively, the maximum influence is approximately 0.0081, while the median influence is only 6.35 × 10⁻⁵. This large gap indicates that influence is very concentrated: a few highly connected nodes (hubs) disproportionately affect the final consensus.

This result is consistent with the theory, since influence is proportional to degree (including the self-loop). Nodes with many connections contribute more to the stationary distribution and therefore have greater impact on the consensus opinion.


## (c) Influence Vector analysis
The influence vector $w$ corresponds to the left-eigenvector of $T$ with eigenvalue 1. In our specific model where nodes listen equally to neighbors and self, the influence of a node is proportional to its augmented degree $\tilde{d}_i$.


In [ ]:
all_influences = np.array(all_influences)
print(f"Maximum Influence: {np.max(all_influences):.6f}")
print(f"Median Influence: {np.median(all_influences):.6f}")

plt.figure(figsize=(10, 6))
plt.hist(all_influences, bins=100, color='skyblue', edgecolor='black')
plt.title("Histogram of Influences across all nodes in all networks")
plt.xlabel("Influence")
plt.ylabel("Frequency (log scale)")
plt.yscale("log")
plt.show()




## (d) Consensus Opinion
The histogram of consensus opinions across all networks is tightly concentrated around 0.5, with an approximate modal value of 0.499.

This result is expected because initial opinions are drawn independently from a uniform distribution between 0 and 1. Since the consensus is a weighted average of these initial opinions, the final value tends to concentrate around the mean of the distribution, which is 0.5.

The small variation across networks is due to differences in network structure (i.e., different influence weights), but these effects are minor relative to the averaging effect.

## (d) Converge Speed 
The histogram of convergence times shows that all networks converge extremely quickly, with t⋆=1 for essentially all cases.

This implies that, after a single update step, the opinions are already very close to the final consensus under the chosen threshold (ϵ=0.1)

This fast convergence is driven by two factors:
1. The averaging process is very strong due to the inclusion of self-loops and equal weighting.
2. The tolerance level (ϵ=0.1) is relatively large, meaning only a coarse level of agreement is required.
If a smaller value of ϵ were used, we would expect larger values of t⋆. 



## (d) Consensus Opinions
Since opinions are initialy $U(0,1)$, the consensus value $p(\infty) = \sum w_i p_i(0)$ is a weighted average of i.i.d. variables. By the Law of Large Numbers, for large networks, this converges to the expected value $0.5$.


In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(consensus_opinions, bins=20, color='salmon', edgecolor='black')
plt.title("Histogram of Consensus Opinions across all networks")
plt.xlabel(r"Consensus Opinion $p(\infty)$")
plt.ylabel("Frequency")
plt.show()

counts, bins = np.histogram(consensus_opinions, bins=20)
mode_idx = np.argmax(counts)
print(f"The modal opinion range is: [{bins[mode_idx]:.4f}, {bins[mode_idx+1]:.4f}]")



## (f) Second Eigenvalue and Convergence

The scatter plot of convergence time t⋆ versus the second eigenvalue shows no visible relationship in this case.

This is because t⋆ is almost always equal to 1, providing no variation in the dependent variable. As a result, it is not possible to observe the expected relationship between the second eigenvalue and convergence speed.

In theory, the second eigenvalue determines how quickly opinions converge: values closer to 1 imply slower convergence. However, due to the coarse convergence threshold and the extremely fast dynamics observed here, this theoretical relationship is not visible in the data.

If a smaller threshold ϵ were used, more variation in t⋆ would likely emerge, making the relationship clearer.      



## (e) & (f) Learning Speed vs. Eigenvalues
The spectral gap $1 - \lambda_2$ determines the rate of convergence. A large $\lambda_2$ (close to 1) indicates slow convergence (high $t^\star$). This is often seen in networks with community structures or bottlenecks.

Lowering $\epsilon$ makes the convergence requirement stricter, which increases $t^\star$ across all networks but maintains the positive correlation with $\lambda_2$.


In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(lambda_2s, t_stars, alpha=0.6, color='green')
plt.title(r"Convergence Time $t^\star$ vs. Second Eigenvalue $\lambda_2$")
plt.xlabel(r"Second Eigenvalue $\lambda_2$")
plt.ylabel(r"Time to convergence $t^\star$")
plt.grid(True, alpha=0.3)
plt.show()


## (g) Friedkin-Johnsen Model
In the FJ model, agents balance their neighbors' opinions with their own innate belief $s_i$. The susceptibility $\alpha_i$ determines how much they listen to their own innate belief.
- When $\alpha \sim U(0.5, 1)$, agents are more **stubborn**, leading to opinions remaining closer to their original beliefs.
- When $\alpha \sim U(0, 1)$, agents are more **susceptible**, and opinions pull closer to the neighborhood average.


In [ ]:
plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
plt.hist(fj_p_stars_u1, bins=50, color='orange', edgecolor='black', alpha=0.7)
plt.title(r"Steady States $p^\star$ ($\alpha \sim U(0,1)$)")
plt.xlabel("Opinion")
plt.ylabel("Frequency")

plt.subplot(1, 2, 2)
plt.hist(fj_p_stars_u2, bins=50, color='purple', edgecolor='black', alpha=0.7)
plt.title(r"Steady States $p^\star$ ($\alpha \sim U(0.5,1)$)")
plt.xlabel("Opinion")
plt.ylabel("Frequency")
plt.show()

print("Mean opinion (alpha U(0,1)): ", np.mean(fj_p_stars_u1))
print("Mean opinion (alpha U(0.5,1)): ", np.mean(fj_p_stars_u2))




## (h)
For the selected network, different strategies for allocating stubbornness (α) were compared under a fixed budget constraint.

1. Uniform allocation results in a total steady-state opinion of approximately 470.55 
2. Heuristic allocation (proportional to initial opinions) improves the result to 629.27
3. Top-p0 allocation (assigning maximum stubbornness to the highest initial opinions) yields the best result, with a total of 712.95

This demonstrates that concentrating stubbornness on nodes with higher initial opinions significantly increases the total final opinion.
The optimal strategy is therefore to: 
1. Assign high stubbornness to individuals with strong positive initial opinions, preventing them from being influenced downward
2. assign low stubbornness to individuals with low initial opinions, allowing them to be influenced upward by others
   
This result aligns with intuition: protecting high-opinion nodes while allowing low-opinion nodes to adapt leads to a higher overall outcome.

### Visualization
Following Abebe et al. (2018), we can optimize $\alpha$ to maximize the sum of opinions. Intuitively, we should make agents with positive/high innate opinions more stubborn (high $\alpha$) and those with low opinions more susceptible (low $\alpha$).


In [ ]:
if chosen_network_data:
    T_net = chosen_network_data['T']
    p_0_net = chosen_network_data['p_0']
    n_net = chosen_network_data['n']
    I_net = sp.eye(n_net)
    budget = n_net * 0.5
    
    def get_sum_opinion(a):
        M = I_net - (I_net - sp.diags(a)).dot(T_net)
        return np.sum(spsolve(M, a * p_0_net))
    
    # Baseline: uniform budget distribution
    alpha_uniform = np.full(n_net, 0.5)
    print("Total opinion with uniform alpha (0.5):", get_sum_opinion(alpha_uniform))
    
    # Heuristic: Be stubborn if your opinion is high, susceptible if low
    alpha_opt = np.clip(p_0_net, 0, 1) # simple mapping
    # Adjust for budget 0.5
    alpha_opt = alpha_opt * (budget / np.sum(alpha_opt))
    alpha_opt = np.clip(alpha_opt, 0, 1)
    
    print("Total opinion with 'stubborn positive' heuristic:", get_sum_opinion(alpha_opt))


# Longer code that spans A through h

In [ ]:
# Results storage
all_influences = []
consensus_opinions = []
t_stars = []
lambda_2s = []
fj_p_stars_u1 = []
fj_p_stars_u2 = []

chosen_network_data = None
chosen_network_name = "Reed98.mat"

np.random.seed(42)

for i, file in enumerate(files):
    if i % 10 == 0:
        print(f"Processing {i}/{len(files)}...")
        
    data = scipy.io.loadmat(os.path.join(folder, file))
    A = data['A']
    G = nx.from_scipy_sparse_array(A)

    # (a) LCC and Matrix Construction
    components = sorted(nx.connected_components(G), key=len, reverse=True)
    lcc_nodes = list(components[0])
    G = G.subgraph(lcc_nodes).copy()
    n = G.number_of_nodes()

    A_lcc = nx.adjacency_matrix(G)
    A_tilde = A_lcc + sp.eye(n)
    degrees_tilde = np.array(A_tilde.sum(axis=1)).flatten()
    
    T = sp.diags(1.0 / degrees_tilde).dot(A_tilde)

    # (c) Influence vector w
    # w is the stationary distribution. Since our transition is w_i = d_tilde_i / sum(d_tilde)
    w = degrees_tilde / degrees_tilde.sum()
    all_influences.extend(w.tolist())

    # (b) Simulate innate opinions U(0,1)
    p_0 = np.random.uniform(0, 1, n)
    p_inf = np.dot(w, p_0)
    consensus_opinions.append(p_inf)

    # (e) Learning Speed t*
    eps = 0.1
    pt = p_0.copy()
    t_star = 0
    for t in range(500):
        # Weighted variance check
        var = np.sum(w * (pt - p_inf)**2) #squared difference of opinions from consensus, weighted by influence
        if var < eps:
            t_star = t #convergence to same opinion within threshold
            break
        pt = T.dot(pt) # sum product of opinions with transition matrix to get next opinions
    else:
        t_star = 500
    t_stars.append(t_star)

    # (f) Second eigenvalue
    try:
        if n > 2:
            vals, _ = eigs(T, k=2, which='LM', tol=1e-2)
            eigenvalues = np.sort(np.abs(vals))
            lambda_2s.append(eigenvalues[-2])
        else:
            lambda_2s.append(0)
    except:
        lambda_2s.append(0)

    # (g) Friedkin-Johnsen steady state
    I = sp.eye(n)
    # alpha ~ U(0,1)
    alpha1 = np.random.uniform(0, 1, n)
    M1 = I - (I - sp.diags(alpha1)).dot(T)
    p_star_1 = spsolve(M1, alpha1 * p_0)
    fj_p_stars_u1.extend(p_star_1.tolist())
        
    # alpha ~ U(0.5,1)
    alpha2 = np.random.uniform(0.5, 1, n)
    M2 = I - (I - sp.diags(alpha2)).dot(T)
    p_star_2 = spsolve(M2, alpha2 * p_0)
    fj_p_stars_u2.extend(p_star_2.tolist())
        
    if file == chosen_network_name:
        chosen_network_data = {'T': T, 'p_0': p_0, 'n': n, 'name': file}

print("Finished processing all networks.")
